# 2 · DPO (Direct Preference Optimization)
The model learns from **pairs of answers: a good one and a bad one**, like a teacher saying *this answer is better than that one*.

## When is DPO worth it?
| Prompting is enough | DPO helps |
|---|---|
| the model already answers well | the model is close, but repeats the same kinds of mistakes |
| mistakes are rare | good and bad answers can be collected (user feedback, corrections) |

## The task: the same shop search box as notebook 1
`black sneakers, under 50 dollars, newest first` → `FIND shoes color=BK max=50 sort=new`

The model keeps the long prompt with every rule. DPO trains away its typical mistakes:
- **A. Prompting only** (the same as notebook 1).
- **B. The same prompt + DPO.**

## Key words
| Word | Meaning |
|---|---|
| **good / bad answer** | for each request: the correct command, and a command with a typical mistake |
| **confidence** | how sure the model is of a whole answer, from 0% to 100% |
| **reference** | a frozen copy of the model from before training, used for comparison |
| **beta** | how far the model may move away from the reference. Small = careful. |
| **epoch** | one pass through all training pairs |

## Step 1 · Load the model
A small open model (0.8 billion parameters). It downloads once.

In [ ]:
import random, warnings, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, logging

warnings.filterwarnings('ignore')             # keep the output readable
logging.set_verbosity_error()

MODEL = 'Qwen/Qwen3.5-0.8B'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map='cuda')

## Step 2 · Create 300 examples, each with a bad answer
The same 300 examples as notebook 1 (same rules, same random seed).
The bad answer copies a **typical mistake**: the color name instead of the code, a missing part, or extra words.

In [ ]:
ITEMS = {'shirt': ['shirt', 't-shirt', 'tee'],
         'shoes': ['shoes', 'sneakers', 'trainers'],
         'jacket': ['jacket', 'coat'],
         'hoodie': ['hoodie', 'sweatshirt']}
COLORS = {'red': 'RD', 'blue': 'BL', 'black': 'BK', 'white': 'WH'}
COLOR_NAMES = {'RD': 'red', 'BL': 'blue', 'BK': 'black', 'WH': 'white'}
PRICES = ['under {} dollars', 'less than ${}', 'below {}', 'max {} bucks']
SORTS = {'price_asc': ['cheapest first', 'lowest price first'],
         'new': ['newest first', 'latest arrivals']}
STARTS = ['', 'show me ', 'I want ', 'looking for ']

def make_example():
    item = random.choice(list(ITEMS))
    color = random.choice(list(COLORS))
    request = random.choice(STARTS) + color + ' ' + random.choice(ITEMS[item])
    command = f'FIND {item} color={COLORS[color]}'
    if random.random() < 0.6:                       # sometimes a price limit
        price = random.choice([20, 30, 50, 80, 100])
        request += ', ' + random.choice(PRICES).format(price)
        command += f' max={price}'
    if random.random() < 0.5:                       # sometimes a sort order
        sort = random.choice(list(SORTS))
        request += ', ' + random.choice(SORTS[sort])
        command += f' sort={sort}'
    return request, command

def bad(command):                                   # a typical mistake
    parts = command.split()
    mistake = random.choice(['color name', 'missing part', 'extra words'])
    if mistake == 'color name':                     # the color name instead of the code
        code = parts[2].removeprefix('color=')
        parts[2] = 'color=' + COLOR_NAMES[code]
    elif mistake == 'missing part' and len(parts) > 3:   # forgets the price or the sort
        parts.pop()
    else:                                           # extra words
        parts = ['Here', 'is', 'the', 'command:'] + parts
    return ' '.join(parts)

random.seed(0)
examples = {}
while len(examples) < 300:
    request, command = make_example()
    examples[request] = command                     # each request only once
examples = list(examples.items())
TRAIN, TEST = examples[:260], examples[260:]

for request, command in TRAIN[:4]:
    print(f'{request}\n   good: {command}\n   bad:  {bad(command)}')

## Step 3 · A. Prompting only
Every rule in the prompt, no training. The same as notebook 1.

**Look for:** which mistakes appear. DPO can only fix the kinds of mistakes shown in the bad answers.

In [ ]:
RULES = '''Convert a shopping request into a search command.
Format: FIND <item> color=<code> max=<price> sort=<order>   (max and sort only if asked)
Items: shirt = shirt, t-shirt, tee | shoes = shoes, sneakers, trainers | jacket = jacket, coat | hoodie = hoodie, sweatshirt
Colors: red=RD, blue=BL, black=BK, white=WH
Sort: price_asc = cheapest first, lowest price first | new = newest first, latest arrivals
Examples:
red tee, below 30 -> FIND shirt color=RD max=30
I want white coat, newest first -> FIND jacket color=WH sort=new
looking for black trainers -> FIND shoes color=BK
Reply with only the command.'''

def long_prompt(request):
    return f'{RULES}\n\nRequest: {request}'

def ask(model, prompt):
    x = tok.apply_chat_template([{'role': 'user', 'content': prompt}], add_generation_prompt=True,
                                return_tensors='pt', return_dict=True).to('cuda')
    y = model.generate(**x, max_new_tokens=30, do_sample=False)
    return tok.decode(y[0][x['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def evaluate(model, make_prompt, show=3):
    model.eval()
    mistakes = []
    for request, command in TEST:
        answer = ask(model, make_prompt(request))
        if answer != command:
            mistakes.append((request, command, answer))
    for request, command, answer in mistakes[:show]:
        print(f'  {request}\n     expected: {command}\n     got:      {answer!r}')
    accuracy = 1 - len(mistakes) / len(TEST)
    print(f'accuracy: {accuracy:.0%}  ({len(TEST) - len(mistakes)} of {len(TEST)} correct)')
    return accuracy

prompt_only = evaluate(model, long_prompt)

## Step 4 · How DPO learns
For one test request: how sure is the model of the good answer, and of the bad one?
DPO raises the first and lowers the second.

In [ ]:
def confidence(model, request, answer):
    model.eval()
    q = tok.apply_chat_template([{'role': 'user', 'content': long_prompt(request)}],
                                add_generation_prompt=True, tokenize=False)
    ids = tok(q + answer, return_tensors='pt').input_ids.to('cuda')
    labels = ids.clone()
    labels[:, :len(tok(q).input_ids)] = -100        # -100 = skip the prompt, score only the answer
    with torch.no_grad():
        loss = model(input_ids=ids, labels=labels).loss   # how wrong the model finds this answer
    return torch.exp(-loss).item()                  # turn it back into 0-100%

request, command = TEST[0]
wrong = bad(command)
good_before = confidence(model, request, command)
bad_before = confidence(model, request, wrong)
print(request)
print(f'  good: {command:45} confidence {good_before:.0%}')
print(f'  bad:  {wrong:45} confidence {bad_before:.0%}')

## Step 5 · Prepare the training
Each row: `prompt` (the long prompt), `chosen` (good) and `rejected` (bad).
The reference is the same model with the add-on switched off, so no second copy is needed.

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

data = Dataset.from_list([
    {'prompt': [{'role': 'user', 'content': long_prompt(r)}],
     'chosen': [{'role': 'assistant', 'content': c}],
     'rejected': [{'role': 'assistant', 'content': bad(c)}]} for r, c in TRAIN])

trainer = DPOTrainer(
    model=model,
    processing_class=tok,
    train_dataset=data,
    peft_config=LoraConfig(r=16, lora_alpha=32, target_modules='all-linear'),   # the small add-on
    args=DPOConfig(beta=0.1, num_train_epochs=1, learning_rate=5e-5, logging_steps=10),   # 1 epoch per call
)
trainer.model.print_trainable_parameters()

## Step 6 · Train, one epoch at a time
Each step: 8 pairs → push the good answers up and the bad ones down → adjust the add-on.
After each epoch, the 40 test requests are checked.

**Look for:** `rewards/accuracies` near 1.0 (the good answer wins in every pair), and the test accuracy going up.

In [ ]:
for epoch in [1, 2, 3]:
    trainer.train()                                 # one pass over the 260 pairs
    print(f'\nafter epoch {epoch}:')
    with_dpo = evaluate(trainer.model, long_prompt)

## Step 7 · Did it work?
The same pair as Step 4, then A and B side by side.
It worked if the good answer gained confidence, the bad one lost it, and B clearly beats A.

In [ ]:
print(f'good answer: confidence {good_before:.0%} -> {confidence(trainer.model, request, command):.0%}')
print(f'bad answer:  confidence {bad_before:.0%} -> {confidence(trainer.model, request, wrong):.0%}')

print(f'\nA. prompting only: {prompt_only:.0%} correct')
print(f'B. prompt + DPO:   {with_dpo:.0%} correct')

if prompt_only >= 0.95:
    print('\nPrompting alone was already enough: no training needed.')
elif with_dpo - prompt_only >= 0.10:
    print('\nIt worked: DPO clearly beat prompting only.')
else:
    print('\nNo clear gain: compare the mistakes in Step 3 with the bad answers in Step 2.')

## Summary
- **Needs:** pairs of a good and a bad answer (here 260)
- **Learns:** to prefer the good answer over the bad one
- **Signs it works:** `rewards/accuracies` near 1.0, the test accuracy goes up
- **Keep in mind:** DPO fixes only the kinds of mistakes shown in the bad answers, and works best when the model is already close